# Cafe Sales Data Cleaning

| Project information | Detail |
|---|---|
| **Author** | Phy Fiq |
| **Project duration** | **3 days** |
| **Programming language** | Python 3 |
| **Main libraries** | pandas, NumPy |
| **Dataset** | [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) |
| **Dataset author** | Ahmed Mohamed (Kaggle) |
| **Rows / columns** | 10,000 / 8 |
| **Notebook** | `cafe_sales_cleaning_en.ipynb` |

This project documents an end-to-end data cleaning workflow for a cafe sales dataset, starting from a deliberately messy raw file and ending with an analysis-ready table. The dataset was designed for cleaning practice and contains values such as `ERROR`, `UNKNOWN`, missing entries, and incorrectly typed columns. The work was completed over **3 days using Python**, with an emphasis on explainable cleaning decisions rather than simply dropping rows.

> The Indonesian version is available at [`cafe_sales_cleaning_id.ipynb`](cafe_sales_cleaning_id.ipynb).

## Table of Contents

1. Introduction
2. Setup
3. Data Overview
4. Data Cleaning
   - 4.1 Column names
   - 4.2 Dirty tokens and data types
   - 4.3 Item imputation
   - 4.4 Categorical columns
   - 4.5 Numeric columns
   - 4.6 Transaction date
5. Validation
6. Results
7. Conclusion
8. Reproducibility and Credits

## 1. Introduction

The goal of this project is to prepare cafe transaction data for downstream analysis. It is not a modeling project; the main objective is to make the data **consistent, correctly typed, and reproducible**. Every cleaning decision is based on the data distribution and relationships between columns, so the result is defensible and easy to review.

Specific objectives:
- Identify dirty values (`ERROR`, `UNKNOWN`, missing entries) in every column.
- Fix column names and data types so they match the actual content.
- Handle missing values with a different strategy for each type of column.
- Validate that `total_spent = quantity x price_per_unit`.
- Save the cleaned dataset as a reusable output.

### Data dictionary

| Column | Description | Expected type |
|---|---|---|
| `transaction_id` | Unique transaction identifier | string |
| `item` | Product purchased | string |
| `quantity` | Number of items purchased | integer |
| `price_per_unit` | Price per item | float |
| `total_spent` | Total transaction value | float |
| `payment_method` | Payment method | string |
| `location` | Order type (in-store or takeaway) | string |
| `transaction_date` | Transaction date | datetime |

## 2. Setup

This notebook uses paths relative to the repository root, so it runs from either the `notebooks/` folder or the project root. There is no Google Colab dependency; users can clone the repository and run the notebook locally.

> Note: code and console outputs stay in English in both language versions for consistency, while all explanations are localized.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Resolve paths so the notebook works from both the repository root and notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "dirty_cafe_sales.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "clean_cafe_sales.csv"

print("Raw data       :", RAW_PATH)
print("Processed data :", PROCESSED_PATH)
print("Raw file exists:", RAW_PATH.exists())

Raw data       : C:\Users\abang\OneDrive\Dokumen\personal-project\cafe-sales-analysis\data\raw\dirty_cafe_sales.csv
Processed data : C:\Users\abang\OneDrive\Dokumen\personal-project\cafe-sales-analysis\data\processed\clean_cafe_sales.csv
Raw file exists: True


## 3. Data Overview

The first step is to load the data and inspect its initial structure: row count, data types, and sample values. It is immediately visible that almost every column is typed as `object`, even though three columns should be numeric and one should be a date. Dirty values also appear as text rather than as true missing values.

In [2]:
df = pd.read_csv(RAW_PATH)
df_clean = df.copy()

print(f"Rows: {df_clean.shape[0]:,} | Columns: {df_clean.shape[1]}")
df_clean.head()

Rows: 10,000 | Columns: 8


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [3]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


## 4. Data Cleaning

### 4.1 Column names

Column names contain spaces and inconsistent capitalization. They are standardized to lowercase with `_` separators so they are easy to use in code and consistent with Python conventions.

In [4]:
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_clean.columns.tolist()

['transaction_id',
 'item',
 'quantity',
 'price_per_unit',
 'total_spent',
 'payment_method',
 'location',
 'transaction_date']

### 4.2 Dirty tokens and data types

The dataset uses several markers for invalid values: `ERROR`, `UNKNOWN`, `N/A`, empty strings, and other variants. Before counting missing values, all markers must be converted to `NaN` so they are not treated as valid categories. Numeric and date columns are then converted with `errors="coerce"`, so unparseable values become `NaN` instead of breaking the pipeline.

In [5]:
DIRTY_TOKENS = ["ERROR", "UNKNOWN", "N/A", "NaN", "nan", "", " ", "None", "null"]

audit = pd.DataFrame({
    "dtype_before": df_clean.dtypes.astype(str),
    "unique_values": df_clean.nunique(),
    "dirty_tokens": [
        df_clean[column].astype(str).isin(DIRTY_TOKENS).sum()
        for column in df_clean.columns
    ],
    "missing_values": df_clean.isna().sum(),
})
audit["dirty_or_missing_pct"] = (
    (audit["dirty_tokens"] + audit["missing_values"]) / len(df_clean) * 100
).round(2)
audit

,dtype_before,unique_values,dirty_tokens,missing_values,dirty_or_missing_pct
transaction_id,str,10000,0,0,0.00
item,str,10,636,333,9.69
quantity,str,7,341,138,4.79
price_per_unit,str,8,354,179,5.33
total_spent,str,19,329,173,5.02
payment_method,str,5,599,2579,31.78
location,str,4,696,3265,39.61
transaction_date,str,367,301,159,4.60


In [6]:
df_clean = df_clean.replace(DIRTY_TOKENS, np.nan)

df_clean["quantity"] = pd.to_numeric(df_clean["quantity"], errors="coerce").astype("Int64")
df_clean["price_per_unit"] = pd.to_numeric(df_clean["price_per_unit"], errors="coerce")
df_clean["total_spent"] = pd.to_numeric(df_clean["total_spent"], errors="coerce")
df_clean["transaction_date"] = pd.to_datetime(df_clean["transaction_date"], errors="coerce")

# Snapshot used later to show the before/after cleaning comparison.
before_missing = df_clean.isna().sum()

df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    10000 non-null  str           
 1   item              9031 non-null   str           
 2   quantity          9521 non-null   Int64         
 3   price_per_unit    9467 non-null   float64       
 4   total_spent       9498 non-null   float64       
 5   payment_method    6822 non-null   str           
 6   location          6039 non-null   str           
 7   transaction_date  9540 non-null   datetime64[us]
dtypes: Int64(1), datetime64[us](1), float64(2), str(4)
memory usage: 634.9 KB


### 4.3 Item imputation

The `item` column has 969 missing values (9.69%). No other column can be used to derive the product type with certainty, so missing values are filled by **random sampling from the observed category distribution**. The eight products in this dataset are almost uniformly distributed, so this preserves the category composition. `seed=42` makes the result reproducible.

In [7]:
item_counts = df_clean["item"].value_counts(dropna=False)
print(item_counts)
print(f"\nMissing items: {df_clean['item'].isna().sum():,} "
      f"({df_clean['item'].isna().mean() * 100:.2f}%)")

item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

Missing items: 969 (9.69%)


In [8]:
item_probabilities = df_clean["item"].value_counts(normalize=True)

rng = np.random.default_rng(42)
mask_item = df_clean["item"].isna()
df_clean.loc[mask_item, "item"] = rng.choice(
    item_probabilities.index.to_numpy(),
    size=mask_item.sum(),
    p=item_probabilities.to_numpy(),
)

print(f"Missing items after imputation: {df_clean['item'].isna().sum()}")
df_clean["item"].value_counts()

Missing items after imputation: 0


item
Juice       1303
Coffee      1294
Salad       1263
Cake        1259
Sandwich    1254
Cookie      1222
Smoothie    1203
Tea         1202
Name: count, dtype: int64

### 4.4 Categorical columns

`payment_method` is missing 31.78% of its values and `location` is missing 39.61%. These shares are too large to drop without losing a lot of data, but there is not enough information to recover the true value per row. Before deciding, I compared mean `total_spent` across categories. With no strong pattern suggesting a deterministic rule, missing values are filled by proportional sampling from the valid category distribution. `transaction_date` is missing only 4.60% and is intentionally **left as missing** because dates cannot be engineered without evidence.

In [9]:
categorical_columns = ["payment_method", "location", "transaction_date"]

pd.DataFrame({
    "missing": df_clean[categorical_columns].isna().sum(),
    "missing_pct": (df_clean[categorical_columns].isna().mean() * 100).round(2),
    "valid_values": df_clean[categorical_columns].nunique(),
})

,missing,missing_pct,valid_values
payment_method,3178,31.78,3
location,3961,39.61,2
transaction_date,460,4.60,365


In [10]:
def spend_by_group(column):
    return (
        df_clean.groupby(column, dropna=False)["total_spent"]
        .agg(count="count", mean="mean", median="median", std="std")
        .round(2)
    )

display(spend_by_group("payment_method"))
display(spend_by_group("location"))

,count,mean,median,std
payment_method,,,,
Cash,2132,9.06,8.0,6.08
Credit Card,2171,9.03,8.0,6.02
Digital Wallet,2187,8.95,8.0,6.01
NaN,3008,8.73,7.5,5.96


,count,mean,median,std
location,,,,
In-store,2869,9.03,8.0,6.04
Takeaway,2872,8.78,7.5,5.96
NaN,3757,8.95,8.0,6.02


In [11]:
def impute_proportional(frame, column, seed=42):
    probabilities = frame[column].value_counts(normalize=True)
    mask = frame[column].isna()
    generator = np.random.default_rng(seed)
    frame.loc[mask, column] = generator.choice(
        probabilities.index.to_numpy(),
        size=mask.sum(),
        p=probabilities.to_numpy(),
    )
    return probabilities

payment_probabilities = impute_proportional(df_clean, "payment_method")
location_probabilities = impute_proportional(df_clean, "location")

print("Distribution used for payment_method:")
print(payment_probabilities.round(4))
print("\nDistribution used for location:")
print(location_probabilities.round(4))

Distribution used for payment_method:
payment_method
Digital Wallet    0.3358
Credit Card       0.3332
Cash              0.3310
Name: proportion, dtype: float64

Distribution used for location:
location
Takeaway    0.5004
In-store    0.4996
Name: proportion, dtype: float64


### 4.5 Numeric columns

For numeric columns, the order of operations determines the output quality:

1. **`price_per_unit`** is filled with the **mode price per product**. Each product has one standard price, so the mode is the most sensible value and avoids fabricated prices.
2. **`quantity`** is derived from `total_spent / price_per_unit` when the result is a whole number between 1 and 5. Remaining values are filled by sampling from the valid quantity distribution.
3. **`total_spent`** is recalculated as `quantity x price_per_unit` for every row. This guarantees 100% consistency and fixes rows whose totals did not match.

As a result, no rows need to be dropped and no zero values are forced into the data.

In [12]:
numeric_columns = ["quantity", "price_per_unit", "total_spent"]

pd.DataFrame({
    "missing": df_clean[numeric_columns].isna().sum(),
    "missing_pct": (df_clean[numeric_columns].isna().mean() * 100).round(2),
})

,missing,missing_pct
quantity,479,4.79
price_per_unit,533,5.33
total_spent,502,5.02


In [13]:
complete_rows = df_clean[numeric_columns].notna().all(axis=1)
consistent_totals = (
    df_clean.loc[complete_rows, "total_spent"]
    == df_clean.loc[complete_rows, "quantity"].astype(float)
    * df_clean.loc[complete_rows, "price_per_unit"]
)

print(f"Complete numeric rows : {complete_rows.sum():,}")
print(f"Consistent totals     : {consistent_totals.sum():,} "
      f"({consistent_totals.mean() * 100:.2f}%)")
print(f"Inconsistent totals   : {(~consistent_totals).sum():,} (to be recalculated)")

Complete numeric rows : 8,544
Consistent totals     : 8,544 (100.00%)
Inconsistent totals   : 0 (to be recalculated)


In [14]:
mode_price_by_item = (
    df_clean.groupby("item")["price_per_unit"]
    .agg(lambda values: values.mode().iloc[0])
)

price_missing_before = df_clean["price_per_unit"].isna().sum()
df_clean["price_per_unit"] = df_clean["price_per_unit"].fillna(
    df_clean["item"].map(mode_price_by_item)
)

print(f"price_per_unit filled with mode : {price_missing_before - df_clean['price_per_unit'].isna().sum():,}")
print(f"price_per_unit still missing    : {df_clean['price_per_unit'].isna().sum():,}")

mode_price_by_item.round(2)

price_per_unit filled with mode : 533
price_per_unit still missing    : 0


item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: price_per_unit, dtype: float64

In [15]:
mask_can_derive = (
    df_clean["quantity"].isna()
    & df_clean["price_per_unit"].notna()
    & df_clean["total_spent"].notna()
)

derived_quantity = (
    df_clean.loc[mask_can_derive, "total_spent"]
    / df_clean.loc[mask_can_derive, "price_per_unit"]
)
is_valid_quantity = (
    derived_quantity.round().between(1, 5)
    & np.isclose(derived_quantity, derived_quantity.round())
)

derived_index = is_valid_quantity[is_valid_quantity].index
df_clean.loc[derived_index, "quantity"] = (
    derived_quantity.loc[derived_index].round().astype("Int64")
)

print(f"Quantity recovered from total/price : {len(derived_index):,}")
print(f"Quantity still missing              : {df_clean['quantity'].isna().sum():,}")

Quantity recovered from total/price : 457
Quantity still missing              : 22


In [16]:
quantity_probabilities = df_clean["quantity"].value_counts(normalize=True).sort_index()
mask_quantity = df_clean["quantity"].isna()

df_clean.loc[mask_quantity, "quantity"] = rng.choice(
    quantity_probabilities.index.astype(int).to_numpy(),
    size=mask_quantity.sum(),
    p=quantity_probabilities.to_numpy(),
)

print("Distribution used for quantity:")
print(quantity_probabilities.round(4))
print(f"\nQuantity still missing: {df_clean['quantity'].isna().sum():,}")
df_clean["quantity"].value_counts().sort_index()

Distribution used for quantity:
quantity
1     0.193
2    0.2061
3    0.1951
4    0.1945
5    0.2113
Name: proportion, dtype: Float64

Quantity still missing: 0


quantity
1    1931
2    2061
3    1951
4    1944
5    2113
Name: count, dtype: Int64

In [17]:
df_clean["total_spent"] = (
    df_clean["quantity"].astype(float) * df_clean["price_per_unit"]
)

print(f"total_spent still missing: {df_clean['total_spent'].isna().sum():,}")
df_clean[numeric_columns].describe().round(2)

total_spent still missing: 0


,quantity,price_per_unit,total_spent
count,10000.0,10000.00,10000.00
mean,3.02,2.95,8.93
std,1.42,1.28,6.00
min,1.0,1.00,1.00
25%,2.0,2.00,4.00
50%,3.0,3.00,8.00
75%,4.0,4.00,12.00
max,5.0,5.00,25.00


### 4.6 Transaction date

A total of 460 transactions (4.60%) have no date. These remain missing because fabricating dates would mislead any time-based analysis. Instead, missing dates are recorded as a data quality finding: the cafe should ensure every transaction is timestamped. The dataset still covers 1 January to 31 December 2023 for rows with a valid date.

In [18]:
date_missing = df_clean["transaction_date"].isna().sum()
print(f"Missing dates : {date_missing:,} ({date_missing / len(df_clean) * 100:.2f}%)")
print(f"Date range    : {df_clean['transaction_date'].min().date()} -> "
      f"{df_clean['transaction_date'].max().date()}")

monthly_transactions = (
    df_clean.dropna(subset=["transaction_date"])
    .assign(month=lambda frame: frame["transaction_date"].dt.to_period("M").astype(str))
    .groupby("month")
    .agg(transactions=("transaction_id", "count"), revenue=("total_spent", "sum"))
    .round(2)
)
monthly_transactions

Missing dates : 460 (4.60%)
Date range    : 2023-01-01 -> 2023-12-31


,transactions,revenue
month,,
2023-01,818,7286.0
2023-02,727,6651.5
2023-03,827,7221.0
2023-04,774,7204.0
2023-05,777,6985.5
2023-06,818,7366.0
2023-07,791,6897.0
2023-08,803,7101.0
2023-09,788,6892.5


## 5. Validation

Before saving, I check four things: remaining missing values, total consistency, duplicates, and final data types. Every column except `transaction_date` is fully populated, `total_spent` matches `quantity x price_per_unit`, and `transaction_id` remains unique across all 10,000 rows.

In [19]:
checks = pd.DataFrame({
    "check": [
        "Duplicate transaction_id",
        "Duplicate rows",
        "Missing item",
        "Missing quantity",
        "Missing price_per_unit",
        "Missing total_spent",
        "Missing transaction_date",
    ],
    "result": [
        df_clean["transaction_id"].duplicated().sum(),
        df_clean.duplicated().sum(),
        df_clean["item"].isna().sum(),
        df_clean["quantity"].isna().sum(),
        df_clean["price_per_unit"].isna().sum(),
        df_clean["total_spent"].isna().sum(),
        df_clean["transaction_date"].isna().sum(),
    ],
})
checks

,check,result
0,Duplicate transaction_id,0
1,Duplicate rows,0
2,Missing item,0
3,Missing quantity,0
4,Missing price_per_unit,0
5,Missing total_spent,0
6,Missing transaction_date,460


In [20]:
final_totals_match = np.isclose(
    df_clean["total_spent"],
    df_clean["quantity"].astype(float) * df_clean["price_per_unit"],
).mean()

print(f"Rows                 : {len(df_clean):,}")
print(f"total_spent matches  : {final_totals_match * 100:.2f}%")
print(f"Unique transaction_id: {df_clean['transaction_id'].nunique():,}")
print("\nFinal data types:")
print(df_clean.dtypes)

Rows                 : 10,000
total_spent matches  : 100.00%
Unique transaction_id: 10,000

Final data types:
transaction_id                 str
item                           str
quantity                     Int64
price_per_unit             float64
total_spent                float64
payment_method                 str
location                       str
transaction_date    datetime64[us]
dtype: object


## 6. Results

The table below summarizes missing values before and after cleaning. A total of 9,622 cells were resolved, while 460 dates were intentionally left missing. The final dataset contains 10,000 rows and 8 columns, with no duplicates and fully consistent transaction totals.

In [21]:
comparison = pd.DataFrame({
    "missing_before": before_missing,
    "missing_after": df_clean.isna().sum(),
})
comparison["fixed"] = comparison["missing_before"] - comparison["missing_after"]
comparison

,missing_before,missing_after,fixed
transaction_id,0,0,0
item,969,0,969
quantity,479,0,479
price_per_unit,533,0,533
total_spent,502,0,502
payment_method,3178,0,3178
location,3961,0,3961
transaction_date,460,460,0


In [22]:
revenue_by_item = (
    df_clean.groupby("item")["total_spent"]
    .agg(transactions="count", revenue="sum", average="mean")
    .sort_values("revenue", ascending=False)
    .round(2)
)
revenue_by_item

,transactions,revenue,average
item,,,
Salad,1263,18414.0,14.58
Sandwich,1254,14915.5,11.89
Smoothie,1203,14246.5,11.84
Juice,1303,11561.0,8.87
Cake,1259,11528.0,9.16
Coffee,1294,8386.5,6.48
Tea,1202,5901.5,4.91
Cookie,1222,4297.5,3.52


In [23]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_PATH, index=False)

print(f"Saved {len(df_clean):,} rows to {PROCESSED_PATH}")
df_clean.head()

Saved 10,000 rows to C:\Users\abang\OneDrive\Dokumen\personal-project\cafe-sales-analysis\data\processed\clean_cafe_sales.csv


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Cash,In-store,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 7. Conclusion

1. The cafe dataset was cleaned without dropping a single row. All 10,000 transactions remain available.
2. Dirty values such as `ERROR`, `UNKNOWN`, and empty-string variants were fully converted and handled.
3. Imputation was proportional and reproducible (`seed=42`), keeping category distributions intact.
4. `total_spent` was recalculated from `quantity x price_per_unit` and validated at 100%, removing inconsistent totals.
5. A total of 460 transaction dates (4.60%) remain missing as a data quality finding instead of being fabricated.
6. Key lesson: cleaning quality depends on understanding relationships between columns. Statistical imputation is still an estimate, so the final dataset should carry that caveat when used for business decisions.

## 8. Reproducibility and Credits

Run the notebook from the repository root with dependencies from `requirements.txt`:

```bash
pip install -r requirements.txt
jupyter notebook notebooks/cafe_sales_cleaning_en.ipynb
```

Original dataset: [Cafe Sales - Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) by **Ahmed Mohamed** on Kaggle. The dataset is used for cleaning practice; ownership remains with its author. The code and documentation in this project were produced for learning portfolio purposes.